# AAT6153 Chapter 2 — Demo
## One-Hot Encoding and the Keras Embedding Layer
**Slides:** One-Hot Encoding (15–18), Word Embeddings (19–21), Keras Embedding Layer (22–24)

In this notebook you will:
1. One-hot encode a sentence with NumPy and with `keras.ops.one_hot`.
2. See why one-hot vectors carry **no notion of similarity**.
3. Use the Keras `Embedding` layer as a **lookup table**, and check that it equals *one-hot × weight matrix*.
4. Use **masking** so that padding is ignored.

*Runtime: CPU is fine for this notebook.*

---
## One-hot encoding with NumPy (slide 16)

In [ ]:
import numpy as np

vocab = ["[PAD]", "[UNK]", "the", "cat", "sat", "on", "mat"]
word_index = {w: i for i, w in enumerate(vocab)}

sentence = "the cat sat on the mat".split()
one_hot = np.zeros((len(sentence), len(vocab)), dtype=int)
for t, word in enumerate(sentence):
    one_hot[t, word_index.get(word, 1)] = 1   # 1 = [UNK]

print(one_hot)
print("Shape:", one_hot.shape, "= (sequence length, vocabulary size)")

## One-hot encoding with Keras (slide 17)

In [ ]:
import keras

ids = np.array([2, 3, 4, 5, 2, 6])            # "the cat sat on the mat"
print(keras.ops.one_hot(ids, num_classes=7))

## Limitation: no notion of similarity (slide 18)
Every pair of different one-hot vectors is **orthogonal**: "cat" is as far from "dog" as from "car".

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

cat, dog, car = np.eye(10000)[[3, 17, 512]]   # one-hot vectors
print(cosine(cat, dog), cosine(cat, car))     # 0.0 0.0
print(f"Sparsity of one vector: {1 - cat.sum() / cat.size:.3%} zeros")

---
## The Embedding layer is a lookup table (slides 22–23)
It stores a weight matrix of shape **(vocabulary size, embedding dimension)**. Each integer index selects one row.

In [ ]:
from keras import layers

embedding = layers.Embedding(input_dim=10000,  # vocabulary size
                             output_dim=8)     # embedding dimension

x = np.array([[2, 15, 7, 0],                   # batch of 2 sequences,
              [9, 4, 0, 0]])                   # 4 token ids each
y = embedding(x)
print(y.shape)                                 # (2, 4, 8)
print(embedding.count_params())                # 10000 x 8 = 80000

In [ ]:
# Lookup == one-hot x weight matrix (slide 22), but without building the huge one-hot vectors
W = embedding.get_weights()[0]                     # (10000, 8)
token_ids = np.array([2, 15, 7])
lookup = embedding(token_ids).numpy()              # (3, 8)
matmul = np.eye(10000)[token_ids] @ W              # (3, 10000) @ (10000, 8)
print("Row 2 of W:        ", W[2].round(3))
print("Embedding of id 2: ", lookup[0].round(3))
print("Max |lookup - one-hot @ W| =", np.abs(lookup - matmul).max())

## Masking (slide 24)
`mask_zero=True` tells downstream layers (e.g. LSTM, GlobalAveragePooling1D) to **skip padded positions**. Index 0 must therefore be reserved for padding.

In [ ]:
masked = layers.Embedding(10000, 8, mask_zero=True)
print(masked.compute_mask(x))

In [ ]:
# Effect of the mask: average pooling with and without it
pool = layers.GlobalAveragePooling1D()
emb_out = masked(x)
print("Plain average over all 4 positions (padding included):\n", emb_out.numpy().mean(axis=1).round(3))
print("GlobalAveragePooling1D with the mask (real tokens only):\n", pool(emb_out).numpy().round(3))
# Keras passes the mask along with emb_out automatically, so the pooling layer skips the padded positions

---
## Try it yourself
1. How many parameters would an `Embedding(20000, 300)` layer have? Check with `count_params()`.
2. Compare the memory needed for a batch of 32 reviews × 200 words as one-hot vectors (vocabulary 20,000, float32) vs as 64-dimensional embeddings.

In [ ]:
# Use this cell for your experiments